In [1]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
all_data = pd.read_excel('all_data-4.xlsx')
all_data['target'] = all_data['probename'].str.extract(r'^(.+?)_probe_\d+$')
all_data['binding_target'] = all_data['sequence_x'].str.translate(str.maketrans('ACGTacgt', 'TGCAtgca')).str[::-1]

print(all_data.columns)
#print(all_data.head())
targets = all_data['target'].unique()

import pandas as pd 
#all_data = pd.read_excel('all_data.xlsx')
#all_data['target'] = all_data['probename'].str.extract(r'^(.+?)_probe_\d+$')

targets = all_data['target'].unique()

# Group by target and calculate mean and std of fold_change
stats = all_data.groupby('target')['fold_change'].agg(
    mean='mean',
    std='std',
    n_probes = 'count',
).reset_index()


Index(['Unnamed: 0', 'probename', 'fold_change', 'p_val', 'sRNA', 'sequence_x',
       'start', 'end', 'contains_GGA', 'contains_AGA', 'sequence_y', 'left',
       'right', 'cut_site', 'Hfq_CLASH_Overlap', 'containsBS', 'log2_fc',
       'gc_percent', 'unnanmed', 'probe_sequence', 'genomic_start',
       'start_in_root', 'end_in_root', 'genomic_end', 'full_target_sequence',
       'full_dot_bracket', 'subset_dot_bracket', 'openness', 'target',
       'binding_target'],
      dtype='object')


In [26]:
import pandas as pd
import numpy as np

def score_window(seq, pwm):
    """Score a single 5-nt window against the PWM."""
    score = 0
    for i, nt in enumerate(seq):
        if nt in pwm and i < len(pwm[nt]):
            score += pwm[nt][i]
        else:
            return np.nan  # handle ambiguous bases gracefully
    return score

def get_best_pwm_score(sequence, pwm, window_size=5):
    """
    Slide a window across the sequence, score each window,
    and return the most favorable (lowest/most negative) delta G.
    """
    sequence = sequence.upper().strip()
    if len(sequence) < window_size:
        return np.nan
    
    scores = [
        score_window(sequence[i:i+window_size], pwm)
        for i in range(len(sequence) - window_size + 1)
    ]
    
    valid_scores = [s for s in scores if not np.isnan(s)]
    return min(valid_scores) if valid_scores else np.nan

pwm = {
    'A': [-1.97, -0.02, -2.78, -1.76, -2.11],
    'C': [ -0.27,  0,  0,  -0.54,  -0.08],
    'G': [ -0.49, -0.04,  -3.42,  -5.10, -1.9],
    'T': [ 0,  -0.16,  -0.09,  0,  0],
}

# Apply to your dataframe
all_data['best_delta_g'] = all_data['binding_target'].apply(
    lambda seq: get_best_pwm_score(seq, pwm)
)
print(all_data.head())

   Unnamed: 0            probename  fold_change     p_val   sRNA  \
0           0   ddlA_5_UTR_probe_4          NaN  0.032694  False   
1           1     Tp2_sRNA_probe_4          NaN  0.013199   True   
2           2  yhbO_5_UTR_probe_21    62.075240  0.037779  False   
3           3   scpA_5_UTR_probe_0    56.437366  0.015661  False   
4           4  fhuA_5_UTR_probe_10    44.255914  0.012728  False   

         sequence_x  start  end  contains_GGA  contains_AGA  ...  \
0  AACGGGGGATTCGCGG     35   50         False         False  ...   
1      CTCGACAGAAGT     58   69         False         False  ...   
2  TCACTTCGTGTCCGGC    181  196         False         False  ...   
3      ATTCAGACGTCG      0   11         False         False  ...   
4  CGCTAACCGCTGTGGC    102  117         False         False  ...   

  start_in_root  end_in_root  genomic_end  \
0            35           50       400988   
1            58           69       122799   
2           181          196      3298892   
3 

In [2]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# 1. Load data
# ─────────────────────────────────────────────
#all_data = pd.read_excel('INTDATA_W_STATS.xlsx')
#all_data['target'] = all_data['name'].str.extract(r'^(.+?)_probe_\d+$')
#all_data.columns = ['probename', 'fold_change','p_val','sRNA','target']
#sig = all_data['p_val'] < 0.05
#all_data = all_data[sig]

# ─────────────────────────────────────────────
# 2. Start simple — work with just ONE target
# ─────────────────────────────────────────────

# Pull out a single target's fold changes as a plain list/series
single_target_name = all_data['target'].unique()[0]
single_target_fc   = all_data.loc[all_data['target'] == single_target_name, 'fold_change']

print(f"Working with target: {single_target_name}")
print(f"Fold changes: {single_target_fc.values}\n")

# ── Basic stats ────────────────────────────────────────────────────────
mean   = single_target_fc.mean()
std    = single_target_fc.std(ddof=1)
median = single_target_fc.median()
MAD    = (single_target_fc - median).abs().median()
Q1     = single_target_fc.quantile(0.25)
Q3     = single_target_fc.quantile(0.75)
IQR    = Q3 - Q1

print(f"  mean={mean:.3f}  std={std:.3f}  median={median:.3f}")
print(f"  Q1={Q1:.3f}  Q3={Q3:.3f}  IQR={IQR:.3f}  MAD={MAD:.3f}\n")

# ── Thresholds ─────────────────────────────────────────────────────────
Z_THRESHOLD    = 2.0
IQR_MULTIPLIER = 1.5
MODZ_THRESHOLD = 3.5

# ── Method A: Z-score ──────────────────────────────────────────────────
z_scores = (single_target_fc - mean) / std if std > 0 else pd.Series(np.zeros(len(single_target_fc)), index=single_target_fc.index)
outlier_z = z_scores.abs() > Z_THRESHOLD

# ── Method B: IQR fence ────────────────────────────────────────────────
iqr_lower  = Q1 - IQR_MULTIPLIER * IQR
iqr_upper  = Q3 + IQR_MULTIPLIER * IQR
outlier_iqr = (single_target_fc < iqr_lower) | (single_target_fc > iqr_upper)

# ── Method C: Modified Z-score ─────────────────────────────────────────
mod_z = 0.6745 * (single_target_fc - median) / MAD if MAD > 0 else pd.Series(np.zeros(len(single_target_fc)), index=single_target_fc.index)
outlier_modz = mod_z.abs() > MODZ_THRESHOLD

# ── Combine ────────────────────────────────────────────────────────────
outlier_any = outlier_z | outlier_iqr | outlier_modz

# ── Build a results dataframe for this one target ──────────────────────
single_target_results = all_data.loc[all_data['target'] == single_target_name].copy()
single_target_results['z_score']          = z_scores
single_target_results['modified_z_score'] = mod_z
single_target_results['iqr_lower']        = iqr_lower
single_target_results['iqr_upper']        = iqr_upper
single_target_results['is_outlier_z']     = outlier_z
single_target_results['is_outlier_iqr']   = outlier_iqr
single_target_results['is_outlier_modz']  = outlier_modz
single_target_results['is_outlier_any']   = outlier_any
single_target_results['outlier_direction'] = np.where(
    ~outlier_any, 'not_outlier',
    np.where(single_target_fc > mean, 'high', 'low')
)

print("Single target results:")
print(single_target_results[['probename', 'fold_change', 'z_score', 'is_outlier_any', 'outlier_direction']].to_string(index=False))

# ─────────────────────────────────────────────
# 3. Now expand — wrap the exact same logic
#    into a function and run it for all targets
# ─────────────────────────────────────────────

def get_outliers_for_target(fc: pd.Series) -> pd.DataFrame:
    """
    Takes a Series of fold changes for a single target.
    Returns a DataFrame with outlier scores and flags,
    one row per probe — index matches the input Series.
    """
    mean   = fc.mean()
    std    = fc.std(ddof=1)
    median = fc.median()
    MAD    = (fc - median).abs().median()
    Q1     = fc.quantile(0.25)
    Q3     = fc.quantile(0.75)
    IQR    = Q3 - Q1

    # Z-score
    z_scores    = (fc - mean) / std if std > 0 else pd.Series(np.zeros(len(fc)), index=fc.index)
    outlier_z   = z_scores.abs() > Z_THRESHOLD

    # IQR fence
    iqr_lower   = Q1 - IQR_MULTIPLIER * IQR
    iqr_upper   = Q3 + IQR_MULTIPLIER * IQR
    outlier_iqr = (fc < iqr_lower) | (fc > iqr_upper)

    # Modified Z-score
    mod_z        = 0.6745 * (fc - median) / MAD if MAD > 0 else pd.Series(np.zeros(len(fc)), index=fc.index)
    outlier_modz = mod_z.abs() > MODZ_THRESHOLD

    # Combine
    outlier_any = outlier_z | outlier_iqr | outlier_modz

    return pd.DataFrame({
        'z_score'          : z_scores,
        'modified_z_score' : mod_z,
        'iqr_lower'        : iqr_lower,
        'iqr_upper'        : iqr_upper,
        'is_outlier_z'     : outlier_z,
        'is_outlier_iqr'   : outlier_iqr,
        'is_outlier_modz'  : outlier_modz,
        'is_outlier_any'   : outlier_any,
        'outlier_direction': np.where(~outlier_any, 'not_outlier',
                             np.where(fc > mean, 'high', 'low'))
    }, index=fc.index)


# ── Run for every target and collect results ───────────────────────────
result_chunks = []

for target_name, group in all_data.groupby('target'):
    outlier_df = get_outliers_for_target(group['fold_change'])
    # Stitch the outlier columns back onto the original rows for this target
    chunk = group.copy()
    chunk = pd.concat([chunk, outlier_df], axis=1)
    result_chunks.append(chunk)

# Stack all targets back into one dataframe
all_data_with_outliers = pd.concat(result_chunks, axis=0).reset_index(drop=True)


# ─────────────────────────────────────────────
# 4. Summary stats per target
# ─────────────────────────────────────────────
stats = all_data_with_outliers.groupby('target').agg(
    mean            = ('fold_change', 'mean'),
    std             = ('fold_change', 'std'),
    n_probes        = ('fold_change', 'count'),
    n_outliers      = ('is_outlier_any', 'sum'),
    n_outliers_high = ('outlier_direction', lambda x: (x == 'high').sum()),
    n_outliers_low  = ('outlier_direction', lambda x: (x == 'low').sum()),
).reset_index()

print(f"\nProcessed {len(stats)} targets.")
print(f"Total outlier probes: {all_data_with_outliers['is_outlier_any'].sum()}\n")
print(stats.to_string(index=False))

# ─────────────────────────────────────────────
# 5. Save to Excel
# ─────────────────────────────────────────────
outliers_only = all_data_with_outliers[all_data_with_outliers['is_outlier_any']].copy()
outliers_only = outliers_only.sort_values(['target', 'fold_change'])

with pd.ExcelWriter('all_data_with_outliers.xlsx', engine='openpyxl') as writer:
    all_data_with_outliers.to_excel(writer, sheet_name='All Probes',    index=False)
    outliers_only.to_excel(writer,          sheet_name='Outliers Only', index=False)
    stats.to_excel(writer,                  sheet_name='Target Stats',  index=False)

print("Saved to all_data_with_outliers-7-6.xlsx")

Working with target: csrB_sRNA
Fold changes: [2.98158957 5.9141932  5.60806905 2.92016999 4.22701559 6.29516719
 6.16256918 4.5896343  4.84295978 0.35350151 5.77844428 8.84414119]

  mean=4.876  std=2.137  median=5.226
  Q1=3.916  Q3=5.976  IQR=2.061  MAD=0.968

Single target results:
         probename  fold_change   z_score  is_outlier_any outlier_direction
 csrB_sRNA_probe_5     2.981590 -0.886866           False       not_outlier
csrB_sRNA_probe_25     5.914193  0.485700           False       not_outlier
csrB_sRNA_probe_17     5.608069  0.342422           False       not_outlier
csrB_sRNA_probe_23     2.920170 -0.915613           False       not_outlier
csrB_sRNA_probe_11     4.227016 -0.303961           False       not_outlier
csrB_sRNA_probe_19     6.295167  0.664009           False       not_outlier
 csrB_sRNA_probe_3     6.162569  0.601949           False       not_outlier
csrB_sRNA_probe_18     4.589634 -0.134242           False       not_outlier
 csrB_sRNA_probe_1     4.84296

In [13]:
import pandas as pd
import numpy as np
UTR_data = pd.read_excel('visualization/all_5_utrs_for_start_codon.xlsx')
UTR_data['target'] = UTR_data['UTR'].str.extract(r'^(.+?)_sequfile$')[0]
utrs = UTR_data
UTR_data['start_codon'] = np.where(
utrs['Direction'] == 'F',
pd.to_numeric(utrs['Genomic Coords start codon location'], errors='coerce') - utrs['Left Genomic Coord'],
pd.to_numeric(utrs['Right Genomic Coord']- utrs['Genomic Coords start codon location'], errors='coerce'))
UTR_data = UTR_data[['target','start_codon']]
all_data = all_data.merge(UTR_data, on='target', how = 'left')
# ── 1. Load & prep ──────────────────────────────────────────────────────────
#all_data = pd.read_excel('all_data.xlsx')

# Extract the target name from probename (everything before "_probe_<number>")
#all_data['target'] = all_data['probename'].str.extract(r'^(.+?)_probe_\d+$')

targets = all_data['target'].unique()
print(f"Found {len(targets)} unique targets.\n")

# ── 2. Per-target summary stats ──────────────────────────────────────────────
stats = all_data.groupby('target')['fold_change'].agg(
    mean='mean',
    std='std',
    n_probes='count',
).reset_index()

# ── 3b. Quartile thresholds per target (with small-n guard) ──────────────────

# Minimum number of probes a target must have before quartile analysis is
# meaningful. Below this, the "top/bottom 25%" label is statistically dubious.
MIN_PROBES_FOR_QUARTILE = 8  # adjust to taste

# Optional: minimum fold_change range a target must span before we bother
# splitting it into quartiles. If all probes are nearly identical, the
# top/bottom labels are noise.
MIN_FOLD_CHANGE_RANGE = 0.5  # adjust to your data's scale

# Flag targets that don't meet the bar — we'll carry these flags forward
# into the labelled dataframe so downstream users know why some probes
# have quartile_group = 'insufficient_data'
target_probe_counts = all_data.groupby('target')['fold_change'].agg(
    n_probes='count',
    fc_range=lambda x: x.max() - x.min()
)

eligible_targets = target_probe_counts[
    (target_probe_counts['n_probes'] >= MIN_PROBES_FOR_QUARTILE) &
    (target_probe_counts['fc_range'] >= MIN_FOLD_CHANGE_RANGE)
].index

ineligible_targets = target_probe_counts[
    ~target_probe_counts.index.isin(eligible_targets)
].copy()

if len(ineligible_targets) > 0:
    print(f"WARNING: {len(ineligible_targets)} targets skipped for quartile analysis:")
    print(ineligible_targets.to_string())
    print()

# Only compute quartile thresholds for eligible targets
quartile_thresholds = (
    all_data[all_data['target'].isin(eligible_targets)]
    .groupby('target')['fold_change']
    .quantile([0.25, 0.75])
    .unstack(level=-1)
    .rename(columns={0.25: 'q25', 0.75: 'q75'})
    .reset_index()
)

# ── 4b. Updated probe classifier that handles ineligible targets ─────────────
all_data = all_data.merge(quartile_thresholds, on='target', how='left')
print(all_data.columns)
def classify_probe(row):
    """
    Classify a probe into a quartile group.

    Returns 'insufficient_data' for targets that didn't meet the minimum
    probe count or fold_change spread threshold, so these are never silently
    mixed in with real top/bottom calls.
    """
    # q25/q75 will be NaN for ineligible targets because they weren't in
    # quartile_thresholds — use that as the ineligibility signal
    if pd.isna(row['q25']):
        return 'insufficient_data'
    if row['fold_change'] <= row['q25']:
        return 'bottom_25'
    elif row['fold_change'] >= row['q75']:
        return 'top_25'
    else:
        return 'middle_50'

all_data['quartile_group'] = all_data.apply(classify_probe, axis=1)

# ── 5. Extract the top-25% and bottom-25% probe subsets ──────────────────────
# These DataFrames give you every probe that sits in the extreme quartiles,
# tagged with which target it belongs to and exactly what the threshold was.

top_probes = (
    all_data[all_data['quartile_group'] == 'top_25']
    .sort_values(['target', 'fold_change'], ascending=[True, False])   # highest first within each target
    .reset_index(drop=True)
)

bottom_probes = (
    all_data[all_data['quartile_group'] == 'bottom_25']
    .sort_values(['target', 'fold_change'], ascending=[True, True])    # lowest first within each target
    .reset_index(drop=True)
)

print(f"Top-25% probes    : {len(top_probes):,} rows across {top_probes['target'].nunique()} targets")
print(f"Bottom-25% probes : {len(bottom_probes):,} rows across {bottom_probes['target'].nunique()} targets")
print()

# ── 6. Per-target summary of extreme probes ──────────────────────────────────
# Useful for a quick sanity-check: how many probes fell into each bucket
# for every target, and what were their mean fold-changes?

extreme_summary = (
    all_data[all_data['quartile_group'] != 'middle_50']   # keep only extremes
    .groupby(['target', 'quartile_group'])['fold_change']
    .agg(
        n_probes_in_group='count',
        mean_fold_change='mean',
        min_fold_change='min',
        max_fold_change='max',
    )
    .reset_index()
    .sort_values(['target', 'quartile_group'])
)

print("Extreme-probe summary (top & bottom 25%) per target:")
print(extreme_summary.to_string(index=False))
print()

# ── 7. Export results ────────────────────────────────────────────────────────
with pd.ExcelWriter('probe_quartile_analysis.xlsx', engine='openpyxl') as writer:

    # Full probe table with quartile labels and thresholds attached
    all_data.to_excel(writer, sheet_name='all_probes_labelled', index=False)

    # Summary stats table (mean, std, n, q25, q75 per target)
    stats.to_excel(writer, sheet_name='target_stats', index=False)

    # Only the high-fold-change probes
    top_probes.to_excel(writer, sheet_name='top_25pct_probes', index=False)

    # Only the low-fold-change probes
    bottom_probes.to_excel(writer, sheet_name='bottom_25pct_probes', index=False)

    # Aggregated extreme-probe counts and means
    extreme_summary.to_excel(writer, sheet_name='extreme_probe_summary', index=False)

print("Results written to probe_quartile_analysis.xlsx")

Found 505 unique targets.

               n_probes   fc_range
target                            
3ETSleuZ_sRNA         1   0.000000
MicL_sRNA             3  16.391685
Nc2_sRNA              5   7.582242
PsrN_sRNA             4  10.422924
SraA_sRNA             3   6.343640
SroA_sRNA             1   0.000000
SroC_sRNA             6   9.008866
SroD_sRNA             2   2.335055
SroE_sRNA             2   2.675102
SroG_sRNA             4   2.396983
Tp2_sRNA              6   7.292691
Tpke11_sRNA           4   6.390668
ackA_5_UTR            7   9.360983
ackA_pta_UTR          4   5.534903
acnA_3_UTR            4   1.496221
adeP_3_UTR            5   6.790905
ahr_3_UTR             3   2.492412
aidB_3_UTR            3   8.533221
aidB_5_UTR            6  18.016588
amyA_5_UTR            7  14.354376
arcA_3_UTR            1   0.000000
arcZ_sRNA             1   0.000000
aroG_3_UTR            6   4.657105
aroG_5_UTR            6   4.383571
arrS_sRNA             3   9.599673
asd_3_UTR             3   3.